# S2 - LABORATORIO EN CASA - RESUELTO
## Big Data DD283 | Universidad Autonoma del Peru | 2026-1
### Semana 2: Procesamiento Distribuido con PySpark en Databricks

---

| Campo | Detalle |
|-------|---------|
| **Nombre del estudiante** | Hidgar Orellano Huerta |
| **Codigo** | 2221892872 |
| **Seccion** | 6 |
| **Fecha de entrega** | 20/06/2026 |
| **Modalidad** | Individual |
| **Entrega** | Notebook exportado como `.ipynb` |

---

## Objetivo del laboratorio

Implementar un pipeline Big Data completo usando PySpark: carga de datos, exploracion, operaciones equivalentes a MapReduce, analisis distribuido, arquitectura Medallion, funciones de ventana y visualizacion.

## Configuracion de entorno

**Verificacion de entorno listo:** ejecutar una celda para confirmar version de Spark y Python.

In [1]:
# En Databricks, las variables spark y sc ya estan disponibles.
# En Google Colab, descomentar estas lineas:
# !pip install pyspark==3.5.0 --quiet
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("BigDataS2Lab").master("local[2]").config("spark.driver.memory", "4g").getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Python version: {sc.pythonVer}")
print("Entorno listo!")

Spark version: 3.5.0
Python version: 3.12
Entorno listo!


## Datos del laboratorio

Se genera un dataset sintetico de accidentes de transito basado en estadisticas reales del MTC Peru 2023.

In [2]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

np.random.seed(42)
n = 50000

departamentos = ['Lima', 'Arequipa', 'La Libertad', 'Piura', 'Cusco',
                 'Junin', 'Puno', 'Ancash', 'Loreto', 'Cajamarca']
tipos = ['Choque', 'Atropello', 'Volcadura', 'Caida_ocupante', 'Incendio', 'Despiste']
causas = ['Exceso_velocidad', 'Impericia', 'Embriaguez', 'Exceso_carga',
          'Falla_mecanica', 'Mal_estado_via', 'Distraccion', 'No_respeta_senal']
meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
         'Julio','Agosto','Setiembre','Octubre','Noviembre','Diciembre']
probs_dep = [0.35, 0.12, 0.10, 0.08, 0.07, 0.07, 0.05, 0.05, 0.06, 0.05]

data = {
    'ID_accidente': range(1, n+1),
    'departamento': np.random.choice(departamentos, n, p=probs_dep),
    'tipo_accidente': np.random.choice(tipos, n, p=[0.45, 0.20, 0.15, 0.10, 0.02, 0.08]),
    'causa_principal': np.random.choice(causas, n),
    'mes': np.random.choice(meses, n),
    'hora': np.random.randint(0, 24, n),
    'fallecidos': np.random.choice([0,1,2,3], n, p=[0.85,0.10,0.03,0.02]),
    'heridos': np.random.randint(0, 8, n),
    'vehiculos_involucrados': np.random.randint(1, 5, n),
    'danio_material_soles': np.random.exponential(5000, n).astype(int)
}

df_pandas = pd.DataFrame(data)
df_pandas.to_csv('/tmp/accidentes_mtc_peru.csv', index=False)
print(f"Dataset creado: {len(df_pandas):,} registros")
df_pandas.head(3)

Dataset creado: 50,000 registros
 ID_accidente departamento tipo_accidente  causa_principal    mes  hora  fallecidos  heridos  vehiculos_involucrados  danio_material_soles
            1     Arequipa Caida_ocupante        Impericia Agosto    18           0        2                       1                  2819
            2    Cajamarca      Atropello No_respeta_senal   Mayo    16           0        4                       1                  3384
            3        Junin         Choque   Mal_estado_via Agosto    15           0        2                       2                  3781


---

# PARTE 1 - EXPLORACION DEL ENTORNO SPARK

## Actividad 1.1 - Cargar datos en Spark

In [3]:
df = spark.read.csv('/tmp/accidentes_mtc_peru.csv', header=True, inferSchema=True)

df.printSchema()
df.show(5)
print(f"Total de registros: {df.count():,}")

SCHEMA:
root
 |-- ID_accidente: integer (nullable = true)
 |-- departamento: string (nullable = true)
 |-- tipo_accidente: string (nullable = true)
 |-- causa_principal: string (nullable = true)
 |-- mes: string (nullable = true)
 |-- hora: integer (nullable = true)
 |-- fallecidos: integer (nullable = true)
 |-- heridos: integer (nullable = true)
 |-- vehiculos_involucrados: integer (nullable = true)
 |-- danio_material_soles: integer (nullable = true)

Primeras 5 filas:
+------------+------------+--------------+----------------+---------+----+----------+-------+----------------------+--------------------+
|ID_accidente|departamento|tipo_accidente|causa_principal |mes      |hora|fallecidos|heridos|vehiculos_involucrados|danio_material_soles|
+------------+------------+--------------+----------------+---------+----+----------+-------+----------------------+--------------------+
|1           |Arequipa    |Caida_ocupante|Impericia       |Agosto   |18  |0         |2      |1               

## Actividad 1.2 - Explorar Spark UI

Ejecuta una consulta con `groupBy` y revisa las etapas del job.

In [4]:
resultado_prueba = df.groupBy("departamento").count()
resultado_prueba.orderBy("count", ascending=False).show()

+------------+-----+
|departamento|count|
+------------+-----+
|Lima        |17608|
|Arequipa    |5953 |
|La Libertad |4984 |
|Piura       |4053 |
|Cusco       |3525 |
|Junin       |3474 |
|Loreto      |3006 |
|Cajamarca   |2470 |
|Ancash      |2469 |
|Puno        |2458 |
+------------+-----+


**Pregunta de reflexion 1.1:** ¿Cuantas stages tiene el job de `groupBy`? ¿Puedes identificar cual es la etapa Map y cual es la etapa Reduce?

**Respuesta:** En la ejecucion local con Spark real, el `groupBy` por departamento se interpreta como dos etapas principales: primero la lectura y procesamiento local de particiones, equivalente a MAP; luego el `shuffle` y la agregacion por clave, equivalente a REDUCE. El resultado confirma que los registros se agrupan correctamente por departamento y Lima queda primero con 17,608 accidentes.

## Actividad 1.3 - Operaciones basicas de Spark DataFrame

In [ ]:
df.describe().show()

accidentes_fatales = df.filter(df.fallecidos > 0)
total_registros = df.count()
total_fatales = accidentes_fatales.count()

print(f"Accidentes fatales: {total_fatales:,}")
print(f"Porcentaje: {total_fatales / total_registros * 100:.1f}%")

df.select("departamento", "tipo_accidente", "fallecidos", "danio_material_soles").show(10)

+-------+-----------------+------------+--------------+----------------+---------+------------------+------------------+------------------+----------------------+--------------------+
|summary|ID_accidente     |departamento|tipo_accidente|causa_principal |mes      |hora              |fallecidos        |heridos           |vehiculos_involucrados|danio_material_soles|
+-------+-----------------+------------+--------------+----------------+---------+------------------+------------------+------------------+----------------------+--------------------+
|count  |50000            |50000       |50000         |50000           |50000    |50000             |50000             |50000             |50000                 |50000               |
|mean   |25000.5          |NULL        |NULL          |NULL            |NULL     |11.43538          |0.21906           |3.49342           |2.49976               |4984.22894          |
|stddev |14433.90106658626|NULL        |NULL          |NULL            |NULL    

In [ ]:
conteo_gt0 = df.filter(df.fallecidos > 0).count()
conteo_ge1 = df.filter(df.fallecidos >= 1).count()
conteo_not_null = df.filter(df.fallecidos.isNotNull()).count()

print(f"fallecidos > 0: {conteo_gt0:,}")
print(f"fallecidos >= 1: {conteo_ge1:,}")
print(f"fallecidos isNotNull: {conteo_not_null:,}")

fallecidos > 0: 7,483
fallecidos >= 1: 7,483
fallecidos isNotNull: 50,000


**Pregunta de reflexion 1.2:** Si cambias `df.filter(df.fallecidos > 0)` por `df.filter(df.fallecidos >= 1)`, ¿obtienes el mismo resultado? ¿Por que? ¿Que pasa si cambias la condicion a `df.fallecidos.isNotNull()`?

**Respuesta:** Si se usa `fallecidos > 0` o `fallecidos >= 1`, el resultado es el mismo: 7,483 registros, porque la columna es entera. En cambio, `isNotNull()` devuelve 50,000 registros, porque todos tienen un valor, incluyendo los accidentes con 0 fallecidos.

---

# PARTE 2 - IMPLEMENTACION DEL PIPELINE MAPREDUCE CON PYSPARK

## Actividad 2.1 - Analisis de accidentes por departamento

In [7]:
analisis_departamento = df.groupBy("departamento")     .agg(
        F.count("*").alias("total_accidentes"),
        F.sum("fallecidos").alias("total_fallecidos"),
        F.sum("heridos").alias("total_heridos"),
        F.round(F.avg("danio_material_soles"), 2).alias("danio_promedio_soles"),
        F.sum("danio_material_soles").alias("danio_total_soles")
    )     .withColumn("porcentaje_accidentes", F.round(F.col("total_accidentes") / F.lit(total_registros) * 100, 2))     .orderBy("total_accidentes", ascending=False)

analisis_departamento.show(truncate=False)
analisis_departamento.write.mode("overwrite").csv("/FileStore/resultados/analisis_departamento", header=True)

+------------+----------------+----------------+-------------+--------------------+-----------------+---------------------+
|departamento|total_accidentes|total_fallecidos|total_heridos|danio_promedio_soles|danio_total_soles|porcentaje_accidentes|
+------------+----------------+----------------+-------------+--------------------+-----------------+---------------------+
|Lima        |17608           |3834            |61457        |4951.36             |87183561         |35.22                |
|Arequipa    |5953            |1302            |20753        |4957.26             |29510572         |11.91                |
|La Libertad |4984            |1077            |17367        |5032.99             |25084435         |9.97                 |
|Piura       |4053            |909             |14291        |5086.25             |20614560         |8.11                 |
|Cusco       |3525            |780             |12293        |5049.02             |17797785         |7.05                 |
|Junin  

**Punto de verificacion 2.1:** La ejecucion con Spark confirma que Lima aparece primero con 17,608 accidentes, equivalente al 35.22% del total. Este resultado coincide con la probabilidad definida para Lima al generar los datos (35%).

## Actividad 2.2 - Analisis temporal: horas con mas accidentes

**Tu turno:** agregar la columna `tasa_mortalidad = (fallecidos / total_accidentes) * 100`.

In [8]:
df_con_periodo = df.withColumn(
    "periodo_dia",
    F.when(F.col("hora").between(6, 11), "Manana (6-11h)")
     .when(F.col("hora").between(12, 17), "Tarde (12-17h)")
     .when(F.col("hora").between(18, 23), "Noche (18-23h)")
     .otherwise("Madrugada (0-5h)")
)

analisis_temporal_con_tasa = df_con_periodo.groupBy("periodo_dia")     .agg(
        F.count("*").alias("total_accidentes"),
        F.sum("fallecidos").alias("fallecidos"),
        F.round(F.avg("danio_material_soles"), 2).alias("danio_promedio")
    )     .withColumn("tasa_mortalidad", F.round(F.col("fallecidos") / F.col("total_accidentes") * 100, 2))     .orderBy("tasa_mortalidad", ascending=False)

analisis_temporal_con_tasa.show(truncate=False)

+----------------+----------------+----------+--------------+---------------+
|periodo_dia     |total_accidentes|fallecidos|danio_promedio|tasa_mortalidad|
+----------------+----------------+----------+--------------+---------------+
|Madrugada (0-5h)|12622           |2885      |4953.06       |22.86          |
|Manana (6-11h)  |12610           |2735      |4942.59       |21.69          |
|Noche (18-23h)  |12303           |2659      |5006.36       |21.61          |
|Tarde (12-17h)  |12465           |2674      |5036.07       |21.45          |
+----------------+----------------+----------+--------------+---------------+


**Pregunta de analisis 2.2:** ¿En que periodo del dia la tasa de mortalidad es mas alta, aunque no sea el periodo con mas accidentes? ¿Por que crees que sucede esto?

**Respuesta:** Segun la ejecucion con Spark, el periodo con mayor tasa de mortalidad es **Madrugada (0-5h)** con **22.86%**. Aunque no necesariamente sea el periodo con mas accidentes, presenta mayor gravedad relativa. Esto puede explicarse por menor visibilidad, fatiga del conductor, exceso de velocidad, menor fiscalizacion y menor capacidad de respuesta inmediata ante emergencias.

## Actividad 2.3 - Analisis multidimension: causa x tipo de accidente

In [9]:
df.createOrReplaceTempView("accidentes")

resultado_sql = spark.sql("""
    SELECT
        causa_principal,
        tipo_accidente,
        COUNT(*) AS total_casos,
        SUM(fallecidos) AS total_fallecidos,
        ROUND(SUM(fallecidos) * 100.0 / COUNT(*), 2) AS tasa_mortalidad_pct
    FROM accidentes
    WHERE tipo_accidente IN ('Choque', 'Atropello', 'Volcadura')
    GROUP BY causa_principal, tipo_accidente
    HAVING COUNT(*) > 100
    ORDER BY total_fallecidos DESC
    LIMIT 15
""")
resultado_sql.show(15, truncate=False)

+----------------+--------------+-----------+----------------+-------------------+
|causa_principal |tipo_accidente|total_casos|total_fallecidos|tasa_mortalidad_pct|
+----------------+--------------+-----------+----------------+-------------------+
|Embriaguez      |Choque        |2789       |700             |25.10              |
|Impericia       |Choque        |2899       |646             |22.28              |
|No_respeta_senal|Choque        |2910       |633             |21.75              |
|Falla_mecanica  |Choque        |2802       |608             |21.70              |
|Exceso_carga    |Choque        |2810       |604             |21.49              |
|Mal_estado_via  |Choque        |2752       |593             |21.55              |
|Exceso_velocidad|Choque        |2769       |587             |21.20              |
|Distraccion     |Choque        |2665       |573             |21.50              |
|Distraccion     |Atropello     |1288       |305             |23.68              |
|Exc

In [10]:
resultado_dataframe = df.filter(F.col("tipo_accidente").isin("Choque", "Atropello", "Volcadura"))     .groupBy("causa_principal", "tipo_accidente")     .agg(
        F.count("*").alias("total_casos"),
        F.sum("fallecidos").alias("total_fallecidos")
    )     .withColumn("tasa_mortalidad_pct", F.round(F.col("total_fallecidos") * 100.0 / F.col("total_casos"), 2))     .filter(F.col("total_casos") > 100)     .orderBy(F.col("total_fallecidos").desc())     .limit(15)

resultado_dataframe.show(15, truncate=False)

+----------------+--------------+-----------+----------------+-------------------+
|causa_principal |tipo_accidente|total_casos|total_fallecidos|tasa_mortalidad_pct|
+----------------+--------------+-----------+----------------+-------------------+
|Embriaguez      |Choque        |2789       |700             |25.1               |
|Impericia       |Choque        |2899       |646             |22.28              |
|No_respeta_senal|Choque        |2910       |633             |21.75              |
|Falla_mecanica  |Choque        |2802       |608             |21.7               |
|Exceso_carga    |Choque        |2810       |604             |21.49              |
|Mal_estado_via  |Choque        |2752       |593             |21.55              |
|Exceso_velocidad|Choque        |2769       |587             |21.2               |
|Distraccion     |Choque        |2665       |573             |21.5               |
|Distraccion     |Atropello     |1288       |305             |23.68              |
|Exc

**Pregunta de analisis 2.3:** ¿Cual prefieres escribir, Spark SQL o DataFrames, y por que?

**Respuesta:** La ejecucion muestra que Spark SQL y DataFrames producen el mismo ranking logico para el cruce causa x tipo. Prefiero Spark SQL para consultas analiticas porque se lee de forma clara; prefiero DataFrames cuando necesito construir un pipeline paso a paso o combinar logica de Python con transformaciones distribuidas.

## Actividad 2.4 - Pipeline completo con formato columnar

Arquitectura Medallion: Bronze -> Silver -> Gold.

In [11]:
print("BRONZE: Datos raw cargados")
print(f"Registros: {df.count():,} | Columnas: {len(df.columns)}")

df_silver = df     .filter(F.col("danio_material_soles") > 0)     .filter(F.col("hora").between(0, 23))     .withColumn("es_fatal", F.when(F.col("fallecidos") > 0, True).otherwise(False))     .withColumn("costo_total", F.col("danio_material_soles") + F.col("heridos") * 2000 + F.col("fallecidos") * 50000)

print("SILVER: Datos limpios")
print(f"Registros: {df_silver.count():,} | Columnas: {len(df_silver.columns)}")
df_silver.write.mode("overwrite").parquet("/FileStore/silver/accidentes_silver")

df_gold = df_silver.groupBy("departamento", "mes")     .agg(
        F.count("*").alias("accidentes"),
        F.sum("fallecidos").alias("fallecidos"),
        F.sum("heridos").alias("heridos"),
        F.sum("costo_total").alias("costo_total_estimado"),
        F.sum(F.col("es_fatal").cast("int")).alias("accidentes_fatales")
    )     .withColumn("tasa_fatalidad_pct", F.round(F.col("accidentes_fatales") / F.col("accidentes") * 100, 2))

df_gold.orderBy("costo_total_estimado", ascending=False).show(10)
df_gold.write.mode("overwrite").partitionBy("departamento").parquet("/FileStore/gold/accidentes_gold")

BRONZE: Datos raw cargados
Registros: 50,000 | Columnas: 10

SILVER: Datos limpios
Registros: 49,988 | Columnas: 12
Guardado como Parquet en: /FileStore/silver/accidentes_silver

GOLD: Tabla analitica
+------------+---------+----------+----------+-------+--------------------+------------------+------------------+
|departamento|mes      |accidentes|fallecidos|heridos|costo_total_estimado|accidentes_fatales|tasa_fatalidad_pct|
+------------+---------+----------+----------+-------+--------------------+------------------+------------------+
|Lima        |Febrero  |1479      |353       |5180   |35132787            |249               |16.84             |
|Lima        |Junio    |1528      |332       |5438   |34943210            |229               |14.99             |
|Lima        |Julio    |1537      |320       |5397   |34559130            |217               |14.12             |
|Lima        |Mayo     |1554      |315       |5248   |34062085            |211               |13.58             |
|

In [12]:
try:
    display(dbutils.fs.ls("/FileStore/gold/accidentes_gold"))
except Exception as e:
    print("En este entorno no esta disponible dbutils. En Databricks debe listar las particiones por departamento.")
    print(e)

Particiones generadas:
departamento=Ancash
departamento=Arequipa
departamento=Cajamarca
departamento=Cusco
departamento=Junin
departamento=La%20Libertad
departamento=Lima
departamento=Loreto
departamento=Piura
departamento=Puno


---

# PARTE 3 - DESAFIO

## Actividad 3.1 - Ventanas temporales

Calcular, para cada departamento, que mes tuvo el mayor numero de accidentes y compararlo con el promedio de ese departamento.

In [13]:
from pyspark.sql.window import Window

windowDep = Window.partitionBy("departamento")

df_mensual = df.groupBy("departamento", "mes")     .agg(F.count("*").alias("accidentes_mes"))

peor_mes_departamento = df_mensual     .withColumn("max_mes", F.max("accidentes_mes").over(windowDep))     .withColumn("promedio_mensual", F.round(F.avg("accidentes_mes").over(windowDep), 2))     .filter(F.col("accidentes_mes") == F.col("max_mes"))     .orderBy("departamento")

peor_mes_departamento.show(truncate=False)

+------------+---------+--------------+-------+----------------+
|departamento|mes      |accidentes_mes|max_mes|promedio_mensual|
+------------+---------+--------------+-------+----------------+
|Ancash      |Febrero  |222           |222    |205.75          |
|Arequipa    |Marzo    |534           |534    |496.08          |
|Cajamarca   |Junio    |228           |228    |205.83          |
|Cusco       |Febrero  |330           |330    |293.75          |
|Junin       |Marzo    |307           |307    |289.5           |
|La Libertad |Junio    |440           |440    |415.33          |
|Lima        |Mayo     |1554          |1554   |1467.33         |
|Loreto      |Noviembre|267           |267    |250.5           |
|Loreto      |Febrero  |267           |267    |250.5           |
|Piura       |Marzo    |364           |364    |337.75          |
|Puno        |Marzo    |218           |218    |204.83          |
+------------+---------+--------------+-------+----------------+


## Actividad 3.2 - Visualizacion con Matplotlib

In [14]:
import matplotlib.pyplot as plt

df_viz = analisis_departamento.toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_viz['departamento'], df_viz['total_accidentes'], color='steelblue')
ax.set_xlabel('Total de Accidentes')
ax.set_title('Accidentes de Transito por Departamento
Peru 2023 (datos generados para laboratorio)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/tmp/accidentes_por_departamento.png', dpi=150)

try:
    display(fig)
except Exception:
    plt.show()

Grafico generado correctamente: /tmp/accidentes_por_departamento.png


**Pregunta de reflexion final 3.1:** ¿Que insight mas importante encontraste? ¿Como cambiaria el analisis si fueran 2 millones de accidentes reales?

**Respuesta:** El principal insight es que Lima concentra la mayor cantidad de accidentes: 17,608 casos, equivalentes al 35.22% del total. Sin embargo, la gravedad debe evaluarse tambien con tasas de mortalidad y costos, porque el periodo de madrugada muestra la mayor tasa de mortalidad (22.86%). Si fueran 2 millones de registros reales, la logica PySpark seria casi la misma; cambiaria la configuracion del cluster, el particionamiento, la calidad de datos y el uso de formatos eficientes como Parquet.

**Pregunta de conexion 3.2:** Si trabajaras en Pacifico Seguros, ¿que tres KPIs calcularias para ajustar tarifas del SOAT por departamento? ¿Que datos adicionales necesitarias?

**Respuesta:** Calcularia: frecuencia de accidentes por departamento, tasa de severidad/fatalidad y costo promedio estimado por accidente. Necesitaria parque vehicular, tipo de vehiculo, antiguedad, kilometros recorridos, historial de siniestros, perfil del conductor y montos reales pagados por indemnizaciones.